In [ ]:
import nltk
import re
import math
from collections import defaultdict
import pandas as pd

# Download necessary NLTK data packages
nltk.download('movie_reviews', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True) # Required for some NLTK tokenizers

from nltk.corpus import movie_reviews, stopwords
from nltk.stem import PorterStemmer

# Initialize our tools
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

## Step 1: Document Collection and Tokenization

### 📚 Theory: The Corpus and Vocabulary
In Information Retrieval (IR), the first step is gathering a **corpus**—a large, structured set of texts. To build a robust Inverted Index and calculate meaningful TF-IDF scores, our corpus must be large enough to capture a diverse vocabulary. A vocabulary of over 2,000 unique words ensures that our Inverse Document Frequency (IDF) calculations have enough statistical variance to properly distinguish between common and rare terms.

### ⚙️ The Collection Process
For this assignment, we are utilizing the **NLTK `movie_reviews` corpus**. 
* **Why this dataset?** It contains thousands of movie reviews categorized by sentiment. It provides a rich, varied vocabulary with natural language syntax, making it ideal for testing text normalization and indexing.
* **Implementation:** Instead of loading the entire 2,000-document corpus (which can slow down manual indexing), we will load a representative subset of 100 documents. This easily exceeds the 2,000 unique word threshold while keeping execution times fast for iterative testing.

> 💡 **Original Talking Point (Data Collection):** 
> *While a larger corpus improves the statistical reliability of IDF calculations, it introduces a computational bottleneck during the indexing phase. Furthermore, using a general-purpose stop-word list on a specialized corpus (like medical or legal texts) might accidentally strip out domain-critical terms that act as 'stop words' in everyday English but are highly informative in context. For a production AI Agent, we would need to generate custom, domain-specific stop-word lists.*

In [10]:
# Load 100 movie reviews to ensure we easily exceed 2,000 unique words
# (The full corpus has 2,000 reviews, but 100 is plenty for this vocab size)
documents = [movie_reviews.raw(fileid) for fileid in movie_reviews.fileids()[:100]]

print(f"Successfully loaded {len(documents)} documents.")
print(f"Preview of Document 0 (first 150 chars):\n{documents[0][:150]}...")

Successfully loaded 100 documents.
Preview of Document 0 (first 150 chars):
plot : two teen couples go to a church party , drink and then drive . 
they get into an accident . 
one of the guys dies , but his girlfriend continue...


## Step 2: Linguistic Processing Pipeline (Normalization & Stemming)

### 📚 Theory: Why Preprocess Text?
Raw text is highly unstructured and "noisy." If we feed raw text directly into an Inverted Index, the system will treat "Run", "run", "running", and "ran" as four completely different terms. This creates a **sparse matrix**, dilutes term frequencies, and severely degrades search recall. 

To solve this, we implement a **Linguistic Processing Pipeline** to reduce dimensionality and standardize the text.

### ⚙️ The Algorithms
Our pipeline applies four sequential transformations:
1. **Normalization (Lowercasing):** Converts all characters to lowercase so that "The" and "the" map to the same token.
2. **Tokenization (Regex Extraction):** We use the regular expression `\b\w+\b` to extract only alphanumeric words. This effectively strips away punctuation, special characters, and formatting noise.
3. **Stop-Word Removal:** Filters out high-frequency, low-information words (e.g., "the", "is", "and") using the NLTK English stop-word list. This drastically reduces the size of our inverted index.
4. **Affix Stripping (Stemming):** We apply the **Porter Stemmer** algorithm. Stemming uses heuristic rules to chop off common suffixes (like "-ing", "-es", "-tion") to reduce words to their morphological root form.

> 💡 **Original Talking Point (Stemming vs. Lemmatization in AI):** 
> *While stemming (like the Porter algorithm) is computationally cheap and highly effective for traditional Boolean and TF-IDF retrieval, it often produces non-linguistic root words (e.g., "university" becomes "univers"). In modern AI Agents and RAG (Retrieval-Augmented Generation) pipelines, we often prefer **lemmatization**—which uses vocabulary and morphological analysis to return actual dictionary roots (e.g., "better" becomes "good"). Lemmatization preserves semantic meaning for Large Language Models (LLMs), even though stemming remains the standard for fast, keyword-based search engines.*

In [13]:
def preprocess(text):
    """
    Tokenizes and normalizes text by:
    1. Lowercasing
    2. Extracting only alphanumeric words (removes punctuation)
    3. Removing English stop words
    4. Applying Porter Stemming (affix stripping)
    """
    # 1. Lowercase
    text = text.lower()
    
    # 2. Tokenize & remove punctuation (keep only words)
    words = re.findall(r'\b\w+\b', text)
    
    # 3. Remove stop words
    words = [w for w in words if w not in stop_words]
    
    # 4. Stemming (Affix stripping)
    words = [stemmer.stem(w) for w in words]
    
    return words

# Test the pipeline on the first document
print("Processed tokens from Doc 0 (first 20):")
print(preprocess(documents[0])[:20])

Processed tokens from Doc 0 (first 20):
['plot', 'two', 'teen', 'coupl', 'go', 'church', 'parti', 'drink', 'drive', 'get', 'accid', 'one', 'guy', 'die', 'girlfriend', 'continu', 'see', 'life', 'nightmar', 'deal']


In [12]:
# Collect all unique words across the entire corpus
unique_vocab = set()
for doc in documents:
    unique_vocab.update(preprocess(doc))

print(f"Total unique words in our processed corpus: {len(unique_vocab)}")

if len(unique_vocab) > 2000:
    print("✅ Rubric Requirement Met: Corpus has more than 2,000 unique words.")
else:
    print("⚠️ Warning: Corpus has fewer than 2,000 unique words. Load more documents.")

Total unique words in our processed corpus: 6795
✅ Rubric Requirement Met: Corpus has more than 2,000 unique words.


# What Is an Inverted Index and When to Use It?

An inverted index is a data structure used to map words (terms) to the documents (or positions within documents) that contain them.
It’s called “inverted” because it flips the normal relationship, instead of listing words in a document, it lists documents for each word.

TERMS --> DOCUMENTS

| Term | Documents |
| ---- | --------- |
| cat  | D1, D3    |
| dog  | D1, D2    |
| eats | D2, D3    |

This structure powers most search engines, allowing quick retrieval of all documents containing a term.


### When to Use an Inverted Index

Use an inverted index when you need fast full-text search or information retrieval tasks such as:

* Search engines (Google, Lucene)

* Document similarity analysis

* Keyword-based filtering or recommender systems

It’s optimal when:

* You have large collections of text documents

* You need to find documents matching keywords quickly

* You can preprocess data (build the index once, query many times)

## How to build that inverted-index code Step by step (plain English)

Below are simple, numbered steps you can follow to implement the exact code in Python. After the steps build your own inverted index code.

Goal

Create a dictionary that maps each word → list of document IDs where that word appears.
Example: 'python' : [0, 2] means the word python appears in documents 0 and 2.


Steps

* Start with your documents

* Make a list where each element is one document (a string).

Example:

In [3]:
documents = [
    "Python is fun",
    "I love programming",
    "Python and Java are languages"
]

Import defaultdict

* defaultdict from collections makes it easy to create a dictionary whose values default to an empty list the first time you use a new key.

Write:

In [4]:
from collections import defaultdict


Create the inverted index data structure

Make a defaultdict(list) so every new word key automatically starts with an empty list.

Write:

In [5]:
inverted_index = defaultdict(list)

Loop over documents with enumerate

* Use enumerate(documents) to get both doc_id (0,1,2,...) and the document text.


Normalize text and split into words

* Convert text to lowercase so Python and python are treated the same.

* Split on whitespace with .split() to get a list of words.

Write another loop inside this same loop to:

Remove duplicate words inside the same document using `set()`

* If a document contains the same word multiple times, we usually want the document ID only once in the list.

* Convert words to set(words) to get unique words for that document.

* This prevents putting the same doc_id into a word’s list multiple times for the same document.

* Add the doc_id to each word’s list

* Loop over the unique words and append the doc_id.

In [6]:
# First loop: Go through every document
    # Split the text into words
    
    # Another Loop: For each word, add the document number
        # set() removes duplicates


In [7]:
inverted_index = dict(inverted_index)


In [8]:
print(inverted_index)
print(inverted_index.get("python", "Word not found"))


{}
Word not found


## Another Example of the Inverted Index

We’ll create a dictionary where:

* The key is a word

* The value is a list of document numbers that contain that word

In [ ]:
# Small dataset of "documents"
documents = [
    "cats and dogs",
    "dogs are cute",
    "I love my cats"
]

In [ ]:
from collections import defaultdict

# Create an empty dictionary
inverted_index = defaultdict(list)

#First loop

    #Second Loop




# Print the result
print(inverted_index)


Now, suppose we want to find which documents contain the word “cats”.

In [ ]:
word_to_search = "cats"
result = inverted_index.get(word_to_search)

print(result)

In [ ]:
print(inverted_index.get("elephant", "Word not found"))

## Variants and Related Models

### Biword (Bigram) Indexes

A biword index stores pairs of consecutive words as entries.

Example:
* "new york city is big"

Biword index entries:
| Term (biword) | Documents |
| ------------- | --------- |
| new_york      | D1        |
| york_city     | D1        |
| city_is       | D1        |
| is_big        | D1        |

Use case:

* Improves phrase search accuracy ("new york" as a phrase)

* Used when you want to support exact phrase matching efficiently

Trade-offs:

* Larger index (more entries)

* Limited to fixed phrase lengths (usually 2-grams)

### Positional Indexes

A positional index stores positions of terms in each document.

Example:

| Term | Documents (positions) |
| ---- | --------------------- |
| new  | D1: [1], D2: [4]      |
| york | D1: [2], D2: [5]      |
| city | D1: [3], D3: [7]      |

Now you can check if “new” and “york” occur adjacently (positions 1 and 2 → "new york").

Use case:

* Needed for phrase queries, proximity search

* Enables queries like “find ‘AI model’ within 3 words of ‘language’”

Trade-offs:

* More storage required (because of position lists)

* More complex query processing

## Term-Document Count Matrix (Document-Term Matrix)

This is a matrix representation (used in ML/NLP) where:

* Rows = documents

* Columns = terms

* Entries = frequency (count) of term in document

Example:

| Document | cat | dog | eats |
| -------- | --- | --- | ---- |
| D1       | 1   | 1   | 0    |
| D2       | 0   | 1   | 1    |
| D3       | 1   | 0   | 1    |

Use case:

* Statistical and machine learning models (text classification, clustering, topic modeling)

* When you need a numerical feature representation of text

Trade-offs:

* Very large and sparse (many zeros)

* Slower for direct lookup; better for ML rather than keyword search

## Bag of Words (BoW) Model

A conceptual model, not a data structure.
It represents text as an unordered collection of words, ignoring grammar and word order.

Example for “new york city” → {new:1, york:1, city:1}

Use case:

* Text classification, sentiment analysis, spam detection

* Simple yet effective baseline representation for machine learning

Trade-offs:

* Loses context and word order

* Cannot handle phrases or proximity queries

* Often combined with weighting (like TF-IDF)


| Concept                         | Output Type                      | Example                             |
| ------------------------------- | -------------------------------- | ----------------------------------- |
| **Bag of Words (BoW)**          | Dictionary per document          | `{'cats': 1, 'dogs': 1, 'love': 1}` |
| **Term–Document Matrix (TDCM)** | Matrix / table for all documents | `[[1,1,1,1,1], [0,0,1,0,0], ...]`   |




| Technique            | Stores                  | Preserves Word Order? | Main Use                  | Pros                   | Cons                      |
| -------------------- | ----------------------- | --------------------- | ------------------------- | ---------------------- | ------------------------- |
| **Inverted Index**   | Term → Docs             | No                     | Keyword search            | Fast lookups           | No phrase info            |
| **Biword Index**     | 2-word phrases → Docs   | Yes (2-word only)       | Phrase search             | Handles common phrases | Larger index              |
| **Positional Index** | Term → (Doc, Positions) | Yes                     | Proximity / Phrase search | Flexible, accurate     | High storage              |
| **Term-Doc Matrix**  | Doc × Term counts       | No                     | ML features               | Numeric, easy for ML   | Sparse, not for retrieval |
| **BoW Model**        | Term frequencies        | No                     | ML / NLP                  | Simple, effective      | Loses order/context       |


# Exercise Instructions 

USe the dataset available in the nltk library.

Preprocess documents using lowercasing, tokenization, and stopword removal.

Build an inverted index using the build_inverted_index function.

Search for words like "girlfriend", "nasa", "baseball", "game".


In [19]:
import nltk
nltk.download('movie_reviews')

from nltk.corpus import movie_reviews

documents = [movie_reviews.raw(fileid) for fileid in movie_reviews.fileids()[:10]]  # first 10 docs
print(documents[0][:200])

plot : two teen couples go to a church party , drink and then drive . 
they get into an accident . 
one of the guys dies , but his girlfriend continues to see him in her life , and has nightmares . 
w


[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\Adecarvalhoalvarezz\AppData\Roaming\nltk_data
[nltk_data]     ...
[nltk_data]   Package movie_reviews is already up-to-date!


In [ ]:
import re
from nltk.corpus import stopwords

# Download stopwords the first time
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def preprocess(text):
    # Lowercase
    # Keep only words
    # Remove stopwords
    return words


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Adecarvalhoalvarezz\AppData\Roaming\nltk_data
[nltk_data]     ...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
from collections import defaultdict

def build_inverted_index(docs, preprocess_fn=None):
    """
    Build an inverted index from a list of documents.
    
    Arguments:
        docs (list of str): List of document texts.
        preprocess_fn (function): Function to preprocess each document (optional).
    
    Returns:
        dict: inverted index {term: [doc_ids]}
    """
    index = defaultdict(list)
    
    for doc_id, doc in enumerate(docs):
        # Apply preprocessing if provided
        if preprocess_fn:
            words = preprocess_fn(doc)
        else:
            words = doc.lower().split()
        
        # Add doc_id to each unique word
        for word in set(words):
            ## add here your transformations
            
    return dict(index)


In [ ]:
# Build inverted index with preprocessing
inverted_index = build_inverted_index(documents, preprocess_fn=preprocess)

# Show example
term_to_search = "girlfriend"
#print your results here

term_to_search = "game"
#print your results here


Documents containing 'girlfriend': [0, 7]
Documents containing 'game': None
